# 🔍 Анализ рынка аналитических вакансий (HH.ru)

**Автор:** Виктор Кобцев  
**Дата создания:** 13.06.2026  
**Версия:** 1.0

---

## 📋 Описание проекта

Автономный ETL-пайплайн для мониторинга вакансий на HH.ru.  
Парсер ежедневно собирает данные, сохраняет исторические "слепки" в локальную базу DuckDB,  
на основе которых строится аналитический дашборд в Apache Superset.

**Цели проекта:**
- Анализ динамики рынка аналитических вакансий
- Мониторинг востребованных навыков и технологий
- Исследование зарплатных диапазонов по регионам и уровням

---

## 🗂️ Структура проекта

```
hh_analitics/
├── data/          # База данных DuckDB
├── notebooks/     # Jupyter ноутбуки
│   └── 01_parser.ipynb
└── logs/          # Логи парсера
```

---

## 🧭 Структура ноутбука

1. Импорт библиотек
2. Настройка параметров
3. Функции (все определения собраны здесь, без выполнения)
4. Этапы выполнения пайплайна (фактические запуски по порядку)

## 1. Импорт библиотек

Загружаем всё необходимое: Selenium + BeautifulSoup для парсинга страниц hh.ru, pandas для обработки данных, DuckDB для хранения, tqdm для прогресс-бара.

In [ ]:
# Импорт всех библиотек: работа с браузером (Selenium), парсинг HTML (BeautifulSoup),
# обработка данных (pandas), хранение (DuckDB), системные утилиты, логирование, случайные задержки и прогресс-бар (tqdm).
import re
import pandas as pd
import duckdb
import os
from datetime import datetime
import time
import random
import subprocess
import sys
import json
import logging
from tqdm.notebook import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from itertools import product


print("Все библиотеки загружены успешно!")
print(f"Pandas версия: {pd.__version__}")
print(f"DuckDB версия: {duckdb.__version__}")
print(f"Текущее время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Настройка параметров

Пути к файлам, базовый URL поиска и параметры сегментации выборки — регион, опыт, формат работы, наличие зарплаты. Их декартово произведение даёт полный список сегментов, которые парсер обходит по очереди.

In [ ]:
# Пути, базовый URL и параметры сегментации выборки.
# Базовый URL для поиска
BASE_URL = "https://hh.ru/search/vacancy"

# Пути
DB_PATH = "../data/hh_vacancies.duckdb"
LOG_PATH = "../logs/parser.log"

# Параметры поиска — декартово произведение сегментов
AREAS = ["1", "113&excluded_area=1"]
EXPERIENCE = ["noExperience", "between1And3", "between3And6", "moreThan6"]
WORK_FORMATS = ["ON_SITE", "REMOTE&work_format=HYBRID&work_format=FIELD_WORK"]
HAS_SALARY = ["true", "false"]

# 32 сегмента = 2 региона × 4 уровня опыта × 2 формата × 2 варианта наличия зарплаты
SEGMENTS = list(product(AREAS, EXPERIENCE, WORK_FORMATS, HAS_SALARY))
# SEGMENTS = [("1", "between1And3", "ON_SITE", "true")]  # тестовый запуск на одном сегменте — раскомментировать для отладки

print(f"Всего сегментов: {len(SEGMENTS)}")
for i, (area, exp, fmt, has_salary) in enumerate(SEGMENTS):
    print(f"  {i+1}. area={area} | exp={exp} | format={fmt} | has_salary={has_salary}")

## 3. Функции

Все функции пайплайна собраны в этом разделе — от запуска браузера и парсинга карточек до сохранения в БД и синхронизации с сервером. Здесь только определения, без реального выполнения — запуски вынесены в раздел 4.

### 3.1 Работа с браузером

In [ ]:
# Создаёт headless-браузер Chrome с настройками, снижающими шанс попасть под антибот-защиту
# hh.ru (кастомный user-agent, размер окна, язык). Используется в каждой сессии парсинга.
def create_driver():
    """Создаёт и настраивает браузер Chrome"""
    
    options = Options()
    options.add_argument("--headless")           # без графического интерфейса
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=ru-RU")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    
    driver = webdriver.Chrome(options=options)
    print("Браузер запущен в фоновом режиме")
    return driver

### 3.2 Извлечение данных из карточки вакансии

In [ ]:
# Три функции обработки одной карточки вакансии:
# get_salary — достаёт сырой текст зарплаты из HTML;
# parse_salary — разбирает его на числа (from/to) и валюту;
# parse_vacancy_card — собирает все поля одной вакансии в словарь.
def get_salary(card):
    """Извлекает сырой текст зарплаты"""
    spans = card.find_all("span")
    for span in spans:
        text = span.get_text(strip=True)
        if "₽" in text and len(text) < 60 and len(span.find_all("span")) == 0:
            return text
    return None

def parse_salary(salary_raw):
    """Парсит зарплату: извлекает все числа, min -> from, max -> to"""
    if not salary_raw:
        return None, None, None
    
    # Определяем валюту
    currency = "RUB" if "₽" in salary_raw else None
    
    # Извлекаем все числа (убираем пробелы внутри чисел типа "15 000")
    numbers = re.findall(r'\d[\d\s]*\d|\d+', salary_raw)
    numbers = [int(n.replace(' ', '').replace(' ', '').replace('\xa0', '')) for n in numbers]
    
    # Фильтруем явно нереалистичные значения (например, год в тексте)
    numbers = [n for n in numbers if 1000 <= n <= 10_000_000]
    
    if not numbers:
        return None, None, currency
    
    salary_from = min(numbers)
    salary_to = max(numbers)
    
    return salary_from, salary_to, currency

def parse_vacancy_card(card):
    """Извлекает данные из одной карточки вакансии"""
    try:
        # Название
        title = card.find("a", {"data-qa": "serp-item__title"})
        title = title.text.strip() if title else None

        # Ссылка
        link = card.find("a", {"data-qa": "serp-item__title"})
        link = link.get("href") if link else None

        # Компания
        company = card.find(attrs={"data-qa": "vacancy-serp__vacancy-employer"})
        company = company.text.strip() if company else None

        # Город
        city = card.find(attrs={"data-qa": "vacancy-serp__vacancy-address"})
        city = city.text.strip() if city else None

        # Опыт
        exp = card.find(attrs={"data-qa": lambda x: x and "work-experience" in x})
        exp = exp.text.strip() if exp else None

        # Удалёнка
        remote = card.find(attrs={"data-qa": lambda x: x and "work-schedule-remote" in x})
        is_remote = True if remote else False

        # Отклик
        responded = card.find(attrs={"data-qa": "vacancy-serp__vacancy_responded"})
        is_applied = True if responded else False

        # Зарплата
        salary_raw = get_salary(card)
        salary_from, salary_to, salary_currency = parse_salary(salary_raw)

        return {
            "title": title,
            "company": company,
            "city": city,
            "experience": exp,
            "is_remote": is_remote,
            "is_applied": is_applied,
            "salary_raw": salary_raw,
            "salary_from": salary_from,
            "salary_to": salary_to,
            "salary_currency": salary_currency,
            "url": link,
            "snapshot_date": datetime.now().strftime("%Y-%m-%d"),
        }

    except Exception as e:
        print(f"Ошибка парсинга карточки: {e}")
        return None

print("Функции парсинга обновлены!")

### 3.3 Сбор вакансий по всем сегментам

In [ ]:
# Обходит все сегменты (регион × опыт × формат × зарплата) и внутри каждого — страницы поиска,
# пока не встретит пустую страницу. Собирает все карточки в один список словарей.
def fetch_vacancies(segments):
    """Собирает вакансии по всем сегментам (регион × опыт × формат × зарплата)"""
    from tqdm.notebook import tqdm
    
    driver = create_driver()
    all_vacancies = []
    
    try:
        with tqdm(total=len(segments), desc="Сбор вакансий", unit="сегм") as pbar:
            for area, experience, work_format, has_salary in segments:
                
                for page in range(40):  # 40 — запас страниц на сегмент; страницы кончаются раньше, цикл прерывается сам через break ниже
                    url = (
                        f"{BASE_URL}"
                        f"?text=аналитик+OR+analyst+OR+analytics"
                        f"&search_field=name"
                        f"&area={area}"
                        f"&experience={experience}"
                        f"&work_format={work_format}"
                        f"&salary=1"
                        f"&only_with_salary={has_salary}"
                        f"&page={page}"
                    )
                    
                    driver.get(url)
                    time.sleep(random.uniform(1, 5))  # случайная пауза после загрузки страницы — фиксированное значение слишком похоже на скрипт для антибот-защиты hh.ru
                    
                    soup = BeautifulSoup(driver.page_source, "lxml")
                    cards = soup.find_all(attrs={"data-qa": "vacancy-serp__vacancy"})
                    
                    if not cards:
                        break
                    
                    for card in cards:
                        vacancy = parse_vacancy_card(card)
                        if vacancy:
                            all_vacancies.append(vacancy)
                    
                    time.sleep(random.uniform(1, 5))  # случайная пауза между страницами — та же причина
                
                pbar.set_postfix({"собрано": len(all_vacancies)})
                pbar.update(1)
    
    finally:
        driver.quit()
    
    print(f"\nИтого собрано: {len(all_vacancies)} вакансий")
    return all_vacancies

### 3.4 Сохранение в DuckDB

In [ ]:
# Пишет собранный датафрейм в DuckDB: новые вакансии (по url) добавляются,
# у уже известных обновляется date_last_seen — так отслеживается, какие вакансии всё ещё активны.
def save_to_db(df, db_path):
    con = duckdb.connect(db_path)
    
    # con.execute("DROP TABLE IF EXISTS vacancies")  # !!!Удаление базы!!!
    
    df = df.drop_duplicates(subset=['url'])
    
    con.execute("""
        CREATE TABLE IF NOT EXISTS vacancies (
            title VARCHAR,
            company VARCHAR,
            city VARCHAR,
            experience VARCHAR,
            is_remote BOOLEAN,
            is_applied BOOLEAN,
            salary_raw VARCHAR,
            salary_from DOUBLE,
            salary_to DOUBLE,
            salary_currency VARCHAR,
            url VARCHAR,
            snapshot_date DATE,
            date_last_seen DATE
        )
    """)
    
    # Новые вакансии — которых ещё нет в базе
    con.execute("""
        INSERT INTO vacancies 
        SELECT *, snapshot_date as date_last_seen FROM df
        WHERE url NOT IN (SELECT url FROM vacancies)
    """)
    
    # Обновляем date_last_seen для уже существующих
    con.execute("""
        UPDATE vacancies
        SET date_last_seen = CURRENT_DATE
        WHERE url IN (SELECT url FROM df)
    """)
    
    count = con.execute("SELECT COUNT(*) FROM vacancies").fetchone()[0]
    today = con.execute("SELECT COUNT(*) FROM vacancies WHERE date_last_seen = CURRENT_DATE").fetchone()[0]
    print(f"Всего записей в базе: {count}")
    print(f"Активных сегодня: {today}")
    
    con.close()
    print("Данные сохранены!")

### 3.5 Логирование

In [ ]:
# Настраивает вывод логов в файл и консоль одновременно — используется во всех ручных запусках раздела 4.
def setup_logger(log_path):
    """Настраивает логгер"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_path, encoding="utf-8"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

### 3.6 Синхронизация с удалённым сервером

In [ ]:
# Константы подключения к удалённому серверу + функции копирования CSV по scp
# и запуска слияния в базу на сервере по ssh. Реальный вызов — в разделе 4.4.
# Для портфолио: реальный адрес сервера заменён плейсхолдером.
LOCAL_CSV = "data/latest_parse.csv"
REMOTE_HOST = "user@your-server-host"
REMOTE_INCOMING = "/opt/hh_analitics/data/incoming/latest_parse.csv"
REMOTE_SYNC_CMD = "cd /opt/hh_analitics && venv/bin/python sync_local_parse.py data/incoming/latest_parse.csv"

def run(cmd, label):
    print(f"{label}...")
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Команда завершилась с ошибкой (код {result.returncode})")
    return result

def sync_to_server():
    run(["scp", LOCAL_CSV, f"{REMOTE_HOST}:{REMOTE_INCOMING}"], "1/2: Копирую CSV на сервер")

    result = run(["ssh", REMOTE_HOST, REMOTE_SYNC_CMD], "2/2: Запускаю слияние в базу на сервере")
    print(result.stdout.strip())

    print("\nГотово! Дашборд обновлён.")

## 4. Этапы выполнения пайплайна

Дальше — фактические запуски по порядку. Каждый этап можно выполнять отдельно.

### 4.1 Проверка браузера

In [ ]:
# Быстрая проверка, что Selenium + Chrome настроены корректно, перед полным запуском парсинга.
driver = create_driver()
print(f"Статус: OK")
driver.quit()
print("Браузер закрыт")

### 4.2 Ручной запуск: парсинг без записи в БД

In [ ]:
# Запускает полный обход всех сегментов и собирает вакансии в DataFrame,
# но НЕ сохраняет в БД — удобно, чтобы проверить результат перед записью.
# Ручной запуск: только парсинг (без записи в БД)
logger = setup_logger(LOG_PATH)
raw_vacancies = fetch_vacancies(SEGMENTS)
df = pd.DataFrame(raw_vacancies)
df = df.dropna(subset=["title"])
logger.info(f"Датафрейм готов: {df.shape[0]} строк")

### 4.3 Экспорт в CSV

In [ ]:
# Сохраняет собранный датафрейм в CSV — этот файл дальше уходит на сервер шагом 4.4.
OUTPUT_DIR = "data"
filename = os.path.join(OUTPUT_DIR, "latest_parse.csv")
df.to_csv(filename, index=False)
print(f"Сохранено: {df.shape[0]} строк -> {filename}")

### 4.4 Синхронизация с сервером

In [ ]:
# ВНИМАНИЕ: реально запускает scp + ssh на удалённый сервер —
# выполнять только когда CSV из шага 4.3 готов и его действительно нужно отправить.
if __name__ == "__main__":
    sync_to_server()

### 4.5 Резервная запись в локальную DuckDB

In [ ]:
# Дублирует сегодняшний df в локальную DuckDB как резервную копию, помимо основного
# пути через CSV + удалённый сервер (шаги 4.3-4.4).
# Ручной запуск: только запись в БД
save_to_db(df, DB_PATH)
logger.info("=== Данные сохранены ===")